# Dephased-IC dataset: parallel generation

Generates recordings on the `single-knob-dephased-ic` branch. Each recording
warm-starts from a state snapshot instead of `h.finitialize(-65)`, which removes
the shared initial condition that ignites the flagship network once at ~4.9 s.

**Run the cells in order.** Cell 2 is a preflight that will tell you if anything
is wrong before you spend compute. Cell 3 is the long one.

### The dephasing works

5-recording pilot, all four gating checks passed:

| check | measured |
|---|---|
| population Vm excursion at 4–6 s | **+0.74 mV** (flagship: +10.30 mV) |
| bursts in the flagship's 4.60–5.34 s band | **0 of 6** |
| mean rate | 0.2927 Hz vs flagship 0.2789 (**+4.9%**) |
| V_rest | −82.65 mV vs flagship −83.31 |

### One known artifact, and why `DISCARD_EXTRA_MS` is not optional

The warm start restores **membrane** state (`v`, hh gates, kA gates, `ko`, sAHP)
but **not synaptic** state. `h.finitialize()` zeroes every synaptic conductance
and resets the depression resource to `R = 1`, so each recording rebuilds
recurrent conductance from zero over ~1 s while running at maximum recurrent
gain. In roughly **1 in 3** recordings that ignites a ~57%-participation burst.

Measured properties:

- **Timing is pinned**: sim 960–1030 ms in every case, i.e. file-clock 0–30 ms
  under the flagship's 1000 ms discard — the first bin of kept data.
- **Not snapshot-locked**: 2 of 4 noise seeds burst on the *same* snapshot, so
  adding snapshots does **not** help.
- **Indistinguishable in shape** from a genuine spontaneous burst
  (see `analysis/dephase_marked_rasters_B_zoom.png`), so nothing downstream
  would flag it.

`DISCARD_EXTRA_MS = 3000` moves the kept window past both the burst and the
post-burst suppression trough. Costs ~5% wall clock and no kept data.

The alternative fix — snapshotting and restoring synaptic state too
(`g_ampa`, `g_nmda`, `R`, inhibitory and noise `g`) — removes the rebuild at
source and needs no discard, but requires a fresh ~2.7 h warm-up run. Worth doing
if anything downstream cares about the first seconds of each recording.

Evidence: `analysis/dephase_marked_rasters_C_diagnostic.png`.

## 1. Configuration

In [ ]:
# ---- what to generate -------------------------------------------------------
N_RECORDINGS = 50          # total recordings (resumable: existing files are skipped)
N_WORKERS    = 5           # concurrent NEURON processes
DURATION_MS  = 60000.0     # kept length per recording, matching the flagship

# ---- voltage recording ------------------------------------------------------
# 'all'   every cell   ~77 MB/recording at 2 ms  (3.8 GB at 50)
# 'probe' subset       ~3 MB/recording at 5 ms   (0.15 GB at 50)   <- default
# 'none'  spikes only
VOLTAGE         = 'probe'
VOLTAGE_PROBE_N = 40
VOLTAGE_DT      = 5.0

# ---- mitigation for the settling burst (REQUIRED, not optional) -------------
# The warm start does not restore synaptic state, so every recording rebuilds
# recurrent conductance from zero with fully un-depressed synapses (R = 1). In
# ~1/3 of recordings that ignites a burst, and MEASURED TIMING IS ALWAYS
# sim 960-1030 ms -- i.e. file-clock 0-30 ms with the default 1000 ms discard.
#
# Measured: the event is NOT snapshot-locked (2 of 4 noise seeds burst on the
# same snapshot), so adding snapshots does NOT fix it. Raising the discard does.
#
# Pilot rate profile in 250 ms bins (file clock, 5 recordings, ratio to the
# 0.2914 Hz steady rate): 3.87x over 0-250 ms, then a suppression trough at
# 0.5-0.7x through ~2.5 s (post-burst sAHP load), settling by ~2.75 s.
# 3000 ms of extra discard puts the kept window at sim t = 4 s, clear of both.
# Costs ~5% wall clock and zero kept data. Set to 0.0 to reproduce the pilot.
DISCARD_EXTRA_MS = 3000.0

print('will generate %d recordings x %.0f s using %d workers'
      % (N_RECORDINGS, DURATION_MS/1000, N_WORKERS))
print('voltage: %s (probe_n=%d, dt=%.1f ms)' % (VOLTAGE, VOLTAGE_PROBE_N, VOLTAGE_DT))
print('discard: %.0f ms flagship + %.0f ms extra = %.0f ms total'
      % (1000.0, DISCARD_EXTRA_MS, 1000.0 + DISCARD_EXTRA_MS))

## 2. Preflight — run this before committing compute

In [ ]:
import os, sys, glob, json, subprocess, time
import numpy as np

REPO = os.path.abspath('..') if os.path.exists(os.path.join('..', 'neuron_simulation')) else os.path.abspath('.')
ANALYSIS = os.path.join(REPO, 'analysis')
OUT_DIR  = os.path.join(REPO, 'notebooks', 'NEURON data parallel', 'dephased_ic')
LIBRARY  = os.path.join(ANALYSIS, 'dephase_state_library.npz')
PY       = sys.executable

ok = True
print('repo      :', REPO)
print('python    :', PY)
print('version   :', sys.version.split()[0])
assert os.path.exists(os.path.join(REPO, 'neuron_simulation')), \
    'could not locate the repo root; open the notebook from notebooks/ or the repo root'

# --- interpreter check: the workers run THIS interpreter, so it must have NEURON.
# Picking a non-NEURON kernel (e.g. 3.12) is the most common failure here.
probe = subprocess.run([PY, '-c', 'import neuron; print(neuron.__version__)'],
                       capture_output=True, text=True)
if probe.returncode == 0:
    print('neuron    : %s  OK' % probe.stdout.strip())
else:
    ok = False
    print('\nFAIL: this kernel cannot import neuron.')
    print('      Select the Python 3.9 interpreter that has NEURON installed')
    print('      (VS Code: Select Kernel -> Python Environments...).')
    print('      Error tail:', probe.stderr.strip().splitlines()[-1:])

# --- compiled mechanisms (kA, DepSyn, AmpaNmda, sAHP, kdyn)
mech = subprocess.run([PY, '-c',
                       'import sys; sys.path.insert(0, r"%s");'
                       'from neuron_simulation.neurons import load_mechanisms;'
                       'load_mechanisms(); print("mechanisms OK")' % REPO],
                      capture_output=True, text=True, cwd=REPO)
if mech.returncode == 0:
    print('mechanisms:', mech.stdout.strip().splitlines()[-1])
else:
    ok = False
    print('\nFAIL: compiled mechanisms not loadable.')
    print('      Build them:  cd neuron_simulation && nrnivmodl mechanisms')
    print('      Error tail:', mech.stderr.strip().splitlines()[-1:])

if not os.path.exists(LIBRARY):
    print('\nFAIL: no state library at', LIBRARY)
    print('      build it first:  python analysis/dephase_snapshot.py')
    ok = False
else:
    lib = np.load(LIBRARY)
    n_snap = lib['g_slow'].shape[0]
    print('\nstate library: %d snapshots x %d cells' % lib['g_slow'].shape)
    print('  snapshot times (s):', [float(t)/1000 for t in lib['snapshot_times_ms']])
    print('  g_slow mean %.5f uS, within-population sd %.5f uS'
          % (lib['g_slow'].mean(), lib['g_slow'].std(axis=1).mean()))
    print('  (a shared-IC start would have g_slow = 0 for every cell)')
    if N_RECORDINGS > n_snap:
        reuse = int(np.ceil(N_RECORDINGS / n_snap))
        print('\n  NOTE: %d recordings, %d snapshots -> each reused up to %dx.'
              % (N_RECORDINGS, n_snap, reuse))
        print('  Measured: the t~0 settling event is NOT snapshot-locked (2 of 4')
        print('  noise seeds burst on the same snapshot), so reuse is acceptable')
        print('  and rebuilding a bigger library would NOT remove it.')
        print('  DISCARD_EXTRA_MS = %.0f is what handles it.' % DISCARD_EXTRA_MS)
        if DISCARD_EXTRA_MS < 2000:
            print('\n  WARNING: DISCARD_EXTRA_MS is below 2000 -- the settling burst')
            print('  lands at sim 960-1030 ms and will be inside your kept window.')

existing = sorted(glob.glob(os.path.join(OUT_DIR, 'recording*.npz')))
print('\nalready on disk: %d recordings in %s' % (len(existing), OUT_DIR))
todo = [r for r in range(N_RECORDINGS)
        if not os.path.exists(os.path.join(OUT_DIR, 'recording%03d.npz' % r))]
print('to generate    : %d  ->  %s%s'
      % (len(todo), todo[:12], ' ...' if len(todo) > 12 else ''))

sim_s = (DURATION_MS + 1000.0 + DISCARD_EXTRA_MS) / 1000.0
solo_min = sim_s * 68.4 / 60.0
eta_h = len(todo) * solo_min / 60.0 / max(1, N_WORKERS)
print('\nmeasured: 68.4x realtime -> ~%.0f min per %.0f s recording, solo' % (solo_min, sim_s))
print('estimated wall clock with %d workers: %.1f h' % (N_WORKERS, eta_h))
mb = {'all': 77.0, 'probe': 3.0, 'none': 0.6}[VOLTAGE]
print('estimated disk: %.2f GB (%s voltage)' % (len(todo)*mb/1024.0, VOLTAGE))
print('\nPREFLIGHT %s' % ('OK' if ok else 'FAILED - fix the above before running cell 3'))

## 3. Generate (the long cell)

Launches `N_WORKERS` subprocesses, each taking a contiguous slice of recording
indices. Resumable — rerun the cell and it skips whatever already exists. Worker
logs go to `analysis/_dephase_nb_w*.log`; the cell polls and prints progress.

In [ ]:
assert ok, 'preflight failed - do not run this cell'

slices, per = [], int(np.ceil(len(todo) / max(1, N_WORKERS)))
for w in range(N_WORKERS):
    chunk = todo[w*per:(w+1)*per]
    if chunk:
        slices.append((w, chunk[0], len(chunk)))
print('worker slices (worker, start, count):', slices)

procs = []
for (w, start, count) in slices:
    log = os.path.join(ANALYSIS, '_dephase_nb_w%d.log' % w)
    cmd = [PY, '-u', os.path.join(ANALYSIS, 'dephase_generate.py'),
           '--start', str(start), '--count', str(count),
           '--duration', str(DURATION_MS),
           '--voltage', VOLTAGE,
           '--voltage-probe-n', str(VOLTAGE_PROBE_N),
           '--voltage-dt', str(VOLTAGE_DT),
           '--discard-extra-ms', str(DISCARD_EXTRA_MS)]
    procs.append((w, subprocess.Popen(cmd, stdout=open(log, 'w'),
                                      stderr=subprocess.STDOUT), log))
    print('launched worker %d: recordings %d..%d' % (w, start, start+count-1))
    time.sleep(20)          # stagger so the workers do not all build at once

t0 = time.time()
while any(p.poll() is None for _, p, _ in procs):
    done = len(glob.glob(os.path.join(OUT_DIR, 'recording*.npz')))
    print('[%5.1f min] %d/%d recordings on disk'
          % ((time.time()-t0)/60, done, N_RECORDINGS), flush=True)
    time.sleep(120)

print('\nall workers exited after %.1f min' % ((time.time()-t0)/60))
bad = False
for w, p, log in procs:
    print('  worker %d exit=%s  (log: %s)' % (w, p.returncode, os.path.basename(log)))
    if p.returncode:
        bad = True
        print('    last lines:')
        print('    ' + '\n    '.join(open(log).read().strip().splitlines()[-6:]))
print('recordings on disk:', len(glob.glob(os.path.join(OUT_DIR, 'recording*.npz'))))
if bad:
    print('\nAt least one worker failed - check its log before using the dataset.')

## 4. Validate

The four gating checks: population Vm has no ~4.9 s excursion, zero bursts in the
flagship's 4.60–5.34 s IC band, mean rate near 0.2789 Hz, V_rest near −83.3 mV.
Also reports how many recordings show the residual t~0 event and whether they
cluster by snapshot.

In [ ]:
r = subprocess.run([PY, '-u', os.path.join(ANALYSIS, 'dephase_validate.py')],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode: print('STDERR:', r.stderr[-2000:])

In [ ]:
# Does the residual t~0 event cluster by snapshot? If it does, snapshot reuse is
# creating a cross-recording aligned artifact and the library needs one snapshot
# per recording.
sys.path.insert(0, REPO)
from neuron_simulation.analysis import detect_network_bursts

rows = []
for p in sorted(glob.glob(os.path.join(OUT_DIR, 'recording*.npz'))):
    d = np.load(p, allow_pickle=True)
    st = [np.atleast_1d(np.asarray(t, float)) for t in d['spike_times']]
    n = len(st)
    b = detect_network_bursts({j: st[j] for j in range(n)}, n, float(d['duration']),
                              participation_threshold=0.35, burn_in_ms=0.0)
    early = [x for x in b if x['start_ms'] < 200.0]
    rows.append(dict(rec=int(d['recording_index']), snap=int(d['snapshot_index']),
                     n_bursts=len(b), early=len(early),
                     rate=sum(len(t) for t in st)/(n*float(d['duration'])/1000.0)))

import collections
by_snap = collections.defaultdict(lambda: [0, 0])
for x in rows:
    by_snap[x['snap']][0] += 1
    by_snap[x['snap']][1] += x['early']
print('t~0 events by snapshot:')
for s in sorted(by_snap):
    tot, early = by_snap[s]
    print('  snapshot %2d: %2d recordings, %2d with a t~0 event (%.0f%%)'
          % (s, tot, early, 100.0*early/max(tot, 1)))
n_early = sum(x['early'] for x in rows)
print('\ntotal: %d of %d recordings show a t~0 event' % (sum(1 for x in rows if x['early']), len(rows)))
print('mean rate %.4f Hz (sd %.4f) vs flagship 0.2789'
      % (np.mean([x['rate'] for x in rows]), np.std([x['rate'] for x in rows])))
if n_early and max(v[1]/max(v[0],1) for v in by_snap.values()) > 0.5:
    print('\n=> the event IS concentrated in particular snapshots: rebuild the library'
          '\n   with one snapshot per recording, or raise DISCARD_EXTRA_MS.')
elif n_early:
    print('\n=> the event is spread across snapshots, so it is noise-driven rather'
          '\n   than snapshot-locked: it does not align across recordings.')
else:
    print('\n=> no t~0 events at all.')

## 5. Next steps once the dataset exists

None of the `single-knob-final-v1` numbers transfer — this branch needs its own
characterization and its own GLM run.

```bash
# operating point (V_rest, sAHP load, ik, [K+]o)
python analysis/measure_characterization.py --duration 21000

# burst windows at the project's 0.35 gate
python analysis/burst_windows_p035.py

# GLM at the shipped operating point; --n-recordings caps the set
python analysis/a1_typing_fix.py --n-recordings 50
```

Expect the mean rate to sit a few percent above the flagship's 0.2789 Hz: the
flagship's rate is depressed by post-IC-burst adaptation this branch no longer
has. The pilot measured 0.2901 Hz excluding the first 2 s, i.e. **+4.0%**.